# Шаг 4. Пульт управления: ползунки меняют картинку

Параметры фильтров (порог, размер окна, контраст) редко угадываются с первого раза.
Удобнее всего подбирать их **интерактивно**: двигать ползунок и сразу видеть результат.
В Colab окна OpenCV не работают, поэтому ползунки делаем библиотекой **ipywidgets**: они появятся прямо под ячейкой.

> **Это версия для Google Colab.** На своём компьютере откройте `04_trackbars.ipynb`: там ползунки OpenCV в отдельном окне.

**Режимы** (ползунок `mode`): 0 — цвет, 1 — серый, 2 — негатив, 3 — порог, 4 — контуры Canny.

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV, ipywidgets и помощники, загружаем фото (уменьшаем до 960 × 540).
2. Пишем функцию `render`, которая по значениям параметров строит картинку:
   1. применяем к **исходной** картинке яркость и контраст, затем размытие;
   2. в зависимости от режима превращаем результат в серый / негатив / порог / контуры;
   3. показываем результат.
3. Привязываем к параметрам функции пять ползунков: яркость, контраст, размытие, режим, порог.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session02")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())

## 1. Импорт и загрузка картинки

`fit(img, 960, 540)` уменьшает фото так, чтобы оно влезло в 960 × 540, сохраняя пропорции.
Небольшая картинка обрабатывается быстрее — ползунки реагируют без задержек.

In [ ]:
import cv2
from ipywidgets import FloatSlider, IntSlider, interact

from common import PIPE, fit, load, show_nb

img = fit(load(PIPE), 960, 540)

## 2. Функция, которая строит картинку

При каждом сдвиге ползунка функция заново строит картинку **с нуля из исходной** `img` — поэтому эффекты не накапливаются,
а сдвиг ползунка назад возвращает прежний вид.

Порядок обработки:
1. `convertScaleAbs(img, alpha, beta)` — контраст и яркость: `v · alpha + beta`, с обрезкой до 0…255.
2. Размытие Гаусса: размер окна должен быть нечётным, поэтому из номера `k` делаем `2k + 1` (1 → 3, 2 → 5, …).
3. Режим: серый (`cvtColor`), негатив (`255 - out`), порог (`threshold` с порогом `t`) или контуры
   (`Canny` с порогами `t/2` и `t` — у Canny нижний порог обычно вдвое меньше верхнего).
4. `show_nb` показывает результат прямо в ноутбуке.

In [ ]:
def render(brightness, contrast, blur, mode, threshold):
    out = cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)
    if blur > 0:
        out = cv2.GaussianBlur(out, (2 * blur + 1, 2 * blur + 1), 0)  # размер окна должен быть нечётным

    gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
    if mode == 1:
        out = gray
    elif mode == 2:
        out = 255 - out
    elif mode == 3:
        _, out = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    elif mode == 4:
        out = cv2.Canny(gray, threshold // 2, threshold)

    show_nb(out)

## 3. Ползунки

`interact(функция, параметр=ползунок, ...)` создаёт ползунки и вызывает функцию при каждом их сдвиге.
Имена параметров должны совпадать с именами аргументов `render`:
- **brightness**: −100…100 — сдвиг яркости (0 = без изменений);
- **contrast**: 0.0…3.0 — множитель контраста (1.0 = без изменений);
- **blur**: 0…15 — номер размера окна (0 = без размытия);
- **mode**: 0…4 — режим показа;
- **threshold**: 0…255 — порог для режимов 3 и 4.

`continuous_update=False` пересчитывает картинку, когда вы **отпустили** ползунок: так она не будет отставать от движения.

In [ ]:
interact(
    render,
    brightness=IntSlider(value=0, min=-100, max=100, continuous_update=False),
    contrast=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, continuous_update=False),
    blur=IntSlider(value=0, min=0, max=15, continuous_update=False),
    mode=IntSlider(value=0, min=0, max=4, continuous_update=False),
    threshold=IntSlider(value=120, min=0, max=255, continuous_update=False),
);

## Попробуйте сами

1. Подберите в режиме 3 (порог) такие яркость, контраст и порог, при которых маркировка читается лучше всего. Запишите значения.
2. Сравните режим 4 (контуры) с размытием 0 и с размытием 3. Почему с размытием контуров меньше?
3. Добавьте шестой режим `5` — поворот на 180° (`cv2.rotate`). Не забудьте поменять `max` у ползунка `mode`.
4. Добавьте ползунок для выбора типа размытия: Гаусс или медиана.